# Tahap 2 — Baseline klasik (CPU, tanpa melatih ulang BERT)
Notebook ini **hanya membaca** file yang sudah ada (indeks `idx_*.npy`, `y_*_v3.npy`, teks, dan prediksi BERT/Hybrid) lalu melatih baseline TF-IDF yang ringan.
Tujuan: menjawab pertanyaan reviewer *"apakah BERT/hybrid benar-benar lebih baik daripada metode sederhana?"*

**Runtime: CPU biasa (tidak perlu GPU). Perkiraan waktu: 5–15 menit.**
Protokol sama dengan model utama: threshold per kelas hanya disetel di set validasi (135), lalu dievaluasi sekali di test (271).

In [ ]:
import os, json, re, random, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')
DRIVE='/content/drive/MyDrive'
DATA4=f'{DRIVE}/JURNAL-DATASET'
OUT4=f'{DRIVE}/JURNAL-OUTPUTS/laptop_jurnal'
PREP4=f'{OUT4}/prep'; PRED4=f'{OUT4}/preds'; S2=f'{OUT4}/stage2'
os.makedirs(S2,exist_ok=True)
def P4(fn):
    p=f'{DATA4}/{fn}'
    return p if os.path.exists(p) else f'{DRIVE}/tesis-dataset 3/{fn}'
CATEGORIES=['Intrusive','Avoidance','Arousal','NegCogMood']
SEED=42; random.seed(SEED); np.random.seed(SEED)
TAG='jurnal_final'
print('setup OK')

## 1. Rekonstruksi train / val / test (identik dengan notebook 11)

In [ ]:
norm=lambda s: re.sub(r'\s+',' ',str(s).lower().strip())
y_va=np.load(f'{PREP4}/y_va_v3.npy'); y_te=np.load(f'{PREP4}/y_te_v3.npy'); y_gold=np.load(f'{PREP4}/y_gold_v3.npy')
idx_tr=np.load(f'{PREP4}/idx_tr_all_v3.npy'); idx_va=np.load(f'{PREP4}/idx_va_v3.npy'); idx_ho=np.load(f'{PREP4}/idx_ho_v3.npy')
texts_all=np.load(f'{PREP4}/texts_all_v3.npy',allow_pickle=True).tolist()
set_test=set(norm(texts_all[i]) for i in idx_ho); set_val=set(norm(texts_all[i]) for i in idx_va)
mask_tr=[i for i in idx_tr if norm(texts_all[i]) not in set_test|set_val]
df_gold=pd.read_csv(f'{DATA4}/data_ahli_FIXED.csv',encoding='utf-8-sig')
df_nat=pd.read_csv(f'{DATA4}/anotasi_natural_2000.csv',encoding='utf-8-sig')
df_b3=pd.read_csv(f'{DATA4}/gold_batch3_577.csv',encoding='utf-8-sig')
df_b4=pd.read_csv(P4('gold_batch4_600.csv'),encoding='utf-8-sig')
df_b5=pd.read_csv(P4('gold_batch5_600.csv'),encoding='utf-8-sig')
nat_norm=df_nat['Segmen_teks'].map(norm); mask_nat=~nat_norm.isin(set_test)
b3_norm=df_b3['Segmen_teks'].map(norm)
mask_b3=~(b3_norm.isin(set(norm(texts_all[i]) for i in mask_tr))|b3_norm.isin(set(nat_norm[mask_nat])))
b4_norm=df_b4['Segmen_teks'].map(norm)
mask_b4=~(b4_norm.isin(set(norm(texts_all[i]) for i in mask_tr))|b4_norm.isin(set(nat_norm[mask_nat]))|b4_norm.isin(set(b3_norm[mask_b3])))
b5_norm=df_b5['Segmen_teks'].map(norm)
mask_b5=~(b5_norm.isin(set(norm(texts_all[i]) for i in mask_tr))|b5_norm.isin(set(nat_norm[mask_nat]))|b5_norm.isin(set(b3_norm[mask_b3]))|b5_norm.isin(set(b4_norm[mask_b4])))
tr_raw=df_gold['Segmen_teks'].iloc[mask_tr].tolist()+df_nat['Segmen_teks'][mask_nat].tolist()+df_b3['Segmen_teks'][mask_b3].tolist()+df_b4['Segmen_teks'][mask_b4].tolist()+df_b5['Segmen_teks'][mask_b5].tolist()
va_raw=df_gold['Segmen_teks'].iloc[idx_va].tolist(); te_raw=df_gold['Segmen_teks'].iloc[idx_ho].tolist()
Y_tr=np.vstack([y_gold[mask_tr],df_nat.loc[mask_nat,CATEGORIES].values.astype(float),df_b3.loc[mask_b3,CATEGORIES].values.astype(float),
                df_b4.loc[mask_b4,CATEGORIES].values.astype(float),df_b5.loc[mask_b5,CATEGORIES].values.astype(float)])
assert len(tr_raw)==5313==len(Y_tr), (len(tr_raw),len(Y_tr))
assert len(va_raw)==135 and len(te_raw)==271
print('train',len(tr_raw),'val',len(va_raw),'test',len(te_raw))

## 2. Fungsi evaluasi (sama dengan protokol utama)

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score, mean_squared_error, mean_absolute_error, accuracy_score
from scipy.stats import pearsonr
def binar(y): return (np.asarray(y)>=0.5).astype(int)
def tune_thr(Yv,Pv):
    grid=np.round(np.arange(0.30,0.701,0.01),2); thr=[]
    for k in range(4):
        sc=[f1_score(binar(Yv)[:,k],(Pv[:,k]>=t).astype(int),zero_division=0) for t in grid]
        thr.append(float(grid[int(np.argmax(sc))]))
    return thr
def evaluate(Yt,Pt,thr):
    Yb=binar(Yt); Pb=(Pt>=np.array(thr)).astype(int)
    auc=[roc_auc_score(Yb[:,k],Pt[:,k]) for k in range(4)]
    dom_ok=(np.argmax(Pt,1)==np.argmax(np.asarray(Yt),1)).mean()
    return {'F1_weighted':f1_score(Yb,Pb,average='weighted',zero_division=0),'F1_macro':f1_score(Yb,Pb,average='macro',zero_division=0),
            'AUC_mean':float(np.mean(auc)),'MSE':mean_squared_error(Yt,Pt),'MAE':mean_absolute_error(Yt,Pt),
            'Pearson_r':pearsonr(np.asarray(Yt).ravel(),np.asarray(Pt).ravel())[0],'Dominant_acc':dom_ok,
            **{f'AUC_{c}':a for c,a in zip(CATEGORIES,auc)}}
# baseline trivial semua-positif (sanity check)
triv=evaluate(y_te,np.ones_like(y_te,dtype=float),[0.5]*4)
print('Trivial all-positive  F1w=%.4f  F1m=%.4f'%(triv['F1_weighted'],triv['F1_macro']))

## 3. Baseline klasik (TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_union
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.svm import LinearSVR, SVR
from sklearn.decomposition import TruncatedSVD
from sklearn.multioutput import MultiOutputRegressor
vec=make_union(TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=2,sublinear_tf=True,max_features=60000),
               TfidfVectorizer(lowercase=True,analyzer='char_wb',ngram_range=(2,5),min_df=3,sublinear_tf=True,max_features=80000))
X_tr=vec.fit_transform(tr_raw); X_va=vec.transform(va_raw); X_te=vec.transform(te_raw)
print('TF-IDF:',X_tr.shape)
clip=lambda P: np.clip(P,0,1)
VAL,TEST={},{}
# (a) Ridge regresi
m=Ridge(alpha=3.0).fit(X_tr,Y_tr); VAL['TFIDF-Ridge']=clip(m.predict(X_va)); TEST['TFIDF-Ridge']=clip(m.predict(X_te))
# (b) Linear SVR per kategori
m=MultiOutputRegressor(LinearSVR(C=0.3,epsilon=0.0,max_iter=5000,random_state=SEED)).fit(X_tr,Y_tr)
VAL['TFIDF-LinearSVR']=clip(m.predict(X_va)); TEST['TFIDF-LinearSVR']=clip(m.predict(X_te))
# (c) Logistic Regression pada label biner (probabilitas)
def fit_lr(Xa,Ya,Xb):
    out=[]
    for k in range(4):
        lr=LogisticRegression(C=2.0,max_iter=2000,class_weight=None).fit(Xa,binar(Ya)[:,k]); out.append(lr.predict_proba(Xb)[:,1])
    return np.stack(out,1)
VAL['TFIDF-LogReg']=fit_lr(X_tr,Y_tr,X_va); TEST['TFIDF-LogReg']=fit_lr(X_tr,Y_tr,X_te)
# (d) TF-IDF -> SVD(300) -> RBF-SVR (analog klasik dari komponen SVR pada hybrid)
svd=TruncatedSVD(300,random_state=SEED).fit(X_tr)
Z_tr,Z_va,Z_te=svd.transform(X_tr),svd.transform(X_va),svd.transform(X_te)
Z_tr/=np.linalg.norm(Z_tr,axis=1,keepdims=True)+1e-9; Z_va/=np.linalg.norm(Z_va,axis=1,keepdims=True)+1e-9; Z_te/=np.linalg.norm(Z_te,axis=1,keepdims=True)+1e-9
pv,pt=[],[]
for k in range(4):
    s=SVR(kernel='rbf',C=1.0,epsilon=0.05,gamma='scale').fit(Z_tr,Y_tr[:,k]); pv.append(s.predict(Z_va)); pt.append(s.predict(Z_te))
VAL['TFIDF-SVD-RBFSVR']=clip(np.stack(pv,1)); TEST['TFIDF-SVD-RBFSVR']=clip(np.stack(pt,1))
print('model dilatih:',list(TEST))

## 4. Prediksi BERT & Hybrid yang sudah ada (hanya dibaca)

In [ ]:
ext={'BERT-only (IndoBERT)':f'{PRED4}/{TAG}_hold_pred_s{SEED}.npy','Hybrid BERT-SVR':f'{PRED4}/{TAG}_hybrid_hold_pred.npy'}
THR={'BERT-only (IndoBERT)':[0.30,0.30,0.30,0.36],'Hybrid BERT-SVR':[0.30,0.30,0.30,0.41]}
for k,v in ext.items():
    if not os.path.exists(v):
        alt=v.replace(f'{TAG}_hold_pred_s{SEED}',f'{TAG}_s{SEED}_hold_pred').replace(f'{TAG}_hybrid_hold_pred',f'{TAG}_hybrid_hold_pred')
        v=alt if os.path.exists(alt) else v
    if os.path.exists(v): TEST[k]=np.load(v); print('dimuat',k,TEST[k].shape)
    else: print('TIDAK ADA:',v)

## 5. Tabel perbandingan di set test (271)

In [ ]:
rows=[]
for name,Pt in TEST.items():
    thr=THR.get(name) or tune_thr(y_va,VAL[name])
    r=evaluate(y_te,Pt,thr); r['Model']=name; r['Thresholds']=thr; rows.append(r)
rows.append({'Model':'Trivial all-positive',**triv,'Thresholds':[0.5]*4})
tab=pd.DataFrame(rows).set_index('Model')
show=['F1_weighted','F1_macro','AUC_mean','MSE','MAE','Pearson_r','Dominant_acc']
display(tab[show].round(4))
display(tab[[f'AUC_{c}' for c in CATEGORIES]+['Thresholds']].round(4))
tab.to_csv(f'{S2}/stage2_baseline_table.csv')
for k,v in TEST.items(): np.save(f'{S2}/test_pred_{re.sub(r"[^A-Za-z0-9]+","_",k)}.npy',v)

## 6. Uji signifikansi vs Hybrid (galat kuadrat per teks, bootstrap berpasangan + Wilcoxon)

In [ ]:
from scipy.stats import wilcoxon
def se_text(P): return ((np.asarray(y_te)-P)**2).mean(1)
ref=TEST.get('Hybrid BERT-SVR'); res=[]
rng=np.random.default_rng(SEED)
if ref is not None:
    for name,Pt in TEST.items():
        if name=='Hybrid BERT-SVR': continue
        d=se_text(Pt)-se_text(ref)   # >0 : hybrid lebih baik
        bs=[d[rng.integers(0,len(d),len(d))].mean() for _ in range(10000)]
        try: p=wilcoxon(d).pvalue
        except Exception: p=np.nan
        res.append({'Model':name,'ΔMSE (model - hybrid)':d.mean(),'CI95_low':np.percentile(bs,2.5),'CI95_high':np.percentile(bs,97.5),'Wilcoxon_p':p,'hybrid_lebih_baik_pada':int((d>0).sum())})
    sig=pd.DataFrame(res).set_index('Model'); display(sig.round(5)); sig.to_csv(f'{S2}/stage2_significance_vs_hybrid.csv')
    print('ΔMSE > 0 dan CI di atas 0 => hybrid signifikan lebih baik daripada baseline tsb.')

## 7. Cara membaca hasil
- Jika baseline TF-IDF **kalah jelas** dari BERT/Hybrid pada AUC dan MSE → klaim "model neural diperlukan" kuat.
- Jika baseline **setara** → tulis jujur di naskah: "metode klasik sudah kompetitif; nilai utama studi adalah korpus dan protokol evaluasi".
- Apa pun hasilnya, **kirim file** `stage2_baseline_table.csv` dan `stage2_significance_vs_hybrid.csv` (di Drive: `JURNAL-OUTPUTS/laptop_jurnal/stage2`) atau screenshot tabelnya, lalu saya masukkan ke naskah (Tabel baru + paragraf Pembahasan).